## SETUP

In [108]:
import math 
import random 
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader 

## I. CONSTANT VARIABLES

In [109]:
SEED = 42 
NUM_USERS = 1000 
NUM_ITEMS = 500 
NUM_CATEGORIES = 10 
MIN_INTERACTIONS = 10 
MAX_INTERACTIONS = 50 
MAX_SEQ_LEN = 30 
BATCH_SIZE = 64
PAD_ID = 0 
VOCAB_SIZE = NUM_ITEMS + 1


In [110]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


## Generate item categories

In [111]:
item_ids = np.arange(1,NUM_ITEMS+1) 
item_categories = np.random.randint(0,NUM_CATEGORIES,size=NUM_ITEMS) 
items_df = pd.DataFrame({'item_id':item_ids,'category_id':item_categories}) 

In [112]:
items_df.head()

,item_id,category_id
0,1,6
1,2,3
2,3,7
3,4,4
4,5,6


## Generate user item interactions

In [113]:
interactions = [] 
for user_id in range(1,NUM_USERS+1): 
    favorite_categories = np.random.choice(
        NUM_CATEGORIES,
        size=2 ,
        replace=False 
    )
    num_interactions = np.random.randint(
        MIN_INTERACTIONS, 
        MAX_INTERACTIONS + 1 
    )

    current_time = 0 

    for _ in range(num_interactions):
        use_favorite = np.random.rand() < 0.8 

        if use_favorite: 
            chosen_category = np.random.choice(favorite_categories)
        else: 
            chosen_category = np.random.randint(0,NUM_CATEGORIES)
        
        candidate_items = items_df[items_df['category_id']==chosen_category]['item_id'].values

        chosen_item = np.random.choice(candidate_items) 

        interactions.append({
            "user_id": user_id,
            "item_id": int(chosen_item),
            "category_id": int(chosen_category),
            "timestamp": current_time
        })

        current_time += 1
interactions_df = pd.DataFrame(interactions)
interactions_df = interactions_df.sort_values(by=['user_id','timestamp']).reset_index(drop=True)

In [114]:
interactions_df.head()

,user_id,item_id,category_id,timestamp
0,1,347,2,0
1,1,77,8,1
2,1,206,3,2
3,1,460,2,3
4,1,240,2,4


## Turn interactions into sequences


In [115]:
user_sequences = interactions_df.groupby('user_id')['item_id'].apply(list).to_dict()

In [116]:
user_sequences[1]

[347,
 77,
 206,
 460,
 240,
 332,
 412,
 317,
 45,
 385,
 103,
 380,
 109,
 353,
 5,
 341,
 184,
 190,
 51,
 390,
 95,
 8,
 256,
 256,
 370,
 104,
 441,
 341]

## Train test split 


In [ ]:
train_user_sequences = {}
test_user_sequences = [] 

for user_id, sequence in user_sequences.items(): 
    if len(sequence) < 2 : continue 

    train_sequence = sequence[:-1] 
    target_item = sequence[-1] 

    train_user_sequences[user_id] = train_sequence 

    test_user_sequences.append({
        "user_id": user_id, 
        "input_sequence": train_sequence, 
        "target_item": target_item
    })

In [118]:
train_user_sequences[1]

[347,
 77,
 206,
 460,
 240,
 332,
 412,
 317,
 45,
 385,
 103,
 380,
 109,
 353,
 5,
 341,
 184,
 190,
 51,
 390,
 95,
 8,
 256,
 256,
 370,
 104,
 441]

## Create Training Samples FOR SASRec


In [119]:
training_samples = [] 

for user_id, sequence in train_user_sequences.items(): 
    for target_index in range(1,len(sequence)): 
        input_sequence = sequence[:target_index] 
        target_item = sequence[target_index] 

        if len(input_sequence) > MAX_SEQ_LEN:
            input_sequence = input_sequence[-MAX_SEQ_LEN:] 
        
        training_samples.append({
            "user_id": user_id, 
            "input_sequence": input_sequence,
            "target_item": target_item
        })
samples_df = pd.DataFrame(training_samples) 

In [120]:
samples_df.head()

,user_id,input_sequence,target_item
0,1,[347],77
1,1,"[347, 77]",206
2,1,"[347, 77, 206]",460
3,1,"[347, 77, 206, 460]",240
4,1,"[347, 77, 206, 460, 240]",332


In [121]:
training_samples[0]

{'user_id': 1, 'input_sequence': [347], 'target_item': 77}

## Create Custom Pytorch Dataset 

In [122]:
class ComiRecDataset(Dataset): 
    def __init__(self,samples,max_seq_len,pad_id = 0):
        self.samples = samples 
        self.max_seq_len = max_seq_len 
        self.pad_id = pad_id 
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,index):
        sample = self.samples[index] 

        input_sequence = sample['input_sequence'] 
        target_item = sample['target_item'] 

        if len(input_sequence) > self.max_seq_len: 
            input_sequence = input_sequence[-self.max_seq_len:]

        padding_length = self.max_seq_len - len(input_sequence) 
        padded_sequence = [self.pad_id] * padding_length + input_sequence 

        input_tensor = torch.tensor(padded_sequence,dtype=torch.long)
        target_tensor = torch.tensor(target_item,dtype=torch.long)

        return input_tensor, target_tensor

In [123]:
train_dataset = ComiRecDataset(training_samples,MAX_SEQ_LEN,PAD_ID)

In [124]:
train_dataset[0]

(tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0, 347]),
 tensor(77))

## Create Dataloader

In [125]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [126]:
input_batch,target_batch = next(iter(train_loader))
print("Input batch shape:",input_batch.shape)
print("Target batch shape:",target_batch.shape)
print("Input first item in batch example:\n",input_batch[0])
print("Target first item in batch example:\n",target_batch[0])

Input batch shape: torch.Size([64, 30])
Target batch shape: torch.Size([64])
Input first item in batch example:
 tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 242, 156, 347,
        388, 208])
Target first item in batch example:
 tensor(299)


## Create Multi-Interest Extractor For ComiRecSA 

### Dim Notations: 
#### D => Embedding Dim 
#### hD => Hidden Dim 
#### B => Batch size 
#### K => Number of Interest 
#### L => Sequence Length 
#### Padding => [B,L] 
#### V => [B,K,D]
#### A => [B,K,L]
#### H => [B,L,D] 


In [127]:
class MultiInterestExtractorSA(nn.Module): 
    def __init__(self,embedding_dim,hidden_dim,num_interests,dropout_rate=0.2): 
        super().__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_interests = num_interests

        self.W1 = nn.Linear(embedding_dim, hidden_dim, bias=False) # D => hD 
        self.W2 = nn.Linear(hidden_dim,num_interests, bias=False) # hD => K

        self.dropout = nn.Dropout(dropout_rate)
    def forward(self,H,padding_mask=None): 
        hidden = torch.tanh(self.W1(H)) #[B,L,hD]
        hidden = self.dropout(hidden) 

        hidden = self.W2(hidden) #[B,L,K] 
        hidden = hidden.transpose(1,2) #[B,K,L] 

        if padding_mask is not None: 
            padding_mask = padding_mask.unsqueeze(1) #[B,1,L] <= [B,L] 
            padding_mask = padding_mask.expand(-1,self.num_interests,-1) #[B,K,L] <= [B,1,L]
            hidden = hidden.masked_fill(padding_mask,float("-inf")) 
        
        A = F.softmax(hidden,dim=-1) #[B,K,L]
        
        V = torch.bmm(A,H) # [B,K,D] <= [B,K,L] * [B,L,D] 
        return V,A 


## Create class ComiRecSAModel 

In [128]:
class ComiRecSAModel(nn.Module): 
    def __init__(self,num_items,max_seq_len,embedding_dim,hidden_dim,num_interests,dropout_rate=0.2,pad_id=0): 
        super().__init__()
        self.num_items = num_items 
        self.max_seq_len = max_seq_len 
        self.embedding_dim = embedding_dim 
        self.hidden_dim = hidden_dim 
        self.num_interests = num_interests 
        self.pad_id = pad_id 
        self.vocab_size = num_items + 1 

        self.item_embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.embedding_dim,
            padding_idx = self.pad_id
        )

        self.positional_embedding = nn.Embedding(
            num_embeddings = self.max_seq_len,
            embedding_dim = self.embedding_dim
        )

        self.dropout = nn.Dropout(dropout_rate) 
        self.layer_norm = nn.LayerNorm(self.embedding_dim) 

        self.interest_extractor = MultiInterestExtractorSA(
            embedding_dim = self.embedding_dim,
            hidden_dim = self.hidden_dim,
            num_interests = self.num_interests,
            dropout_rate = dropout_rate
        )
    def forward(self,input_sequences):
        # Input: input_sequences [B,L]
        device = input_sequences.device
        batch_size, seq_len = input_sequences.size() 

        positions = torch.arange(seq_len, device = device)  #[L]
        positions = positions.unsqueeze(0) # [1,L] <= [L]
        positions = positions.expand(batch_size,-1) #[B,L] <= [1,L] 

        item_emb = self.item_embedding(input_sequences) #[B,L,D]
        pos_emb = self.positional_embedding(positions) #[B,L,D] 

        H = item_emb + pos_emb #[B,L,D]
        H = self.dropout(H)
        H = self.layer_norm(H) #[B,L,D] 

        padding_mask = input_sequences.eq(self.pad_id) #[B,L] 
        V,A = self.interest_extractor(H,padding_mask) #[B,K,D],[B,K,L]
        
        return V,A
    def compute_loss(self, input_sequences, target_items):  
        # Input: input_sequences [B,L], target_items [B]
        V, A = self.forward(input_sequences) #[B,K,D],[B,K,L] 
        target_emb = self.item_embedding(target_items) #[B,D]

        # 1. Compute the dot product between target_emb and each interest vector in V
        target_emb = target_emb.unsqueeze(1) #[B,1,D] <= [B,D]
        target_emb = target_emb.expand(-1,self.num_interests,-1) #[B,K,D] <= [B,1,D]
        interest_scores = torch.sum(V * target_emb, dim=-1) #[B,K] <= sum([B,K,D] * [B,K,D], dim=-1)

        # 2. Select the best interest vector that has the highest score for each sample/user 
        best_interest_idx = torch.argmax(interest_scores, dim=-1) #[B] <= argmax([B,K], dim=-1)
        batch_indices = torch.arange(V.size(0), device=V.device) #[B] 
        best_interest = V[batch_indices,best_interest_idx,:] #[B,D] <= V[[B],[B],:] 

        # 3. Compute the loss by dot product between best_interest and all item embeddings
        all_item_emb = self.item_embedding.weight #[num_items+1,D] 
        all_item_emb = all_item_emb.T #[D,num_items+1]
        logits = best_interest @ all_item_emb #[B,num_items+1] <= [B,D] @ [D,num_items+1] 
        logits[:,self.pad_id] = float("-inf")  # Mask the padding index
        loss = F.cross_entropy(logits, target_items) # scalar <= cross_entropy([B,num_items+1],[B])

        return loss


## II. CONSTANT VARIABLES FOR TRAINING

In [129]:
EMBEDDING_DIM = 64
DROPOUT_RATE = 0.2
NUM_EPOCHS = 50
HIDDEN_DIM = EMBEDDING_DIM*4
NUM_INTERESTS = 6

## Initialize SASRec Model


In [130]:
model = ComiRecSAModel(
    num_items = NUM_ITEMS,
    max_seq_len = MAX_SEQ_LEN,
    embedding_dim = EMBEDDING_DIM,
    hidden_dim = HIDDEN_DIM,
    num_interests = NUM_INTERESTS,
    dropout_rate = DROPOUT_RATE,
    pad_id = PAD_ID
)

## Test Forward with 1 Batch

In [131]:
V, A = model(input_batch) #[B,K,D],[B,K,L]
print("Interest vectors shape:",V.shape)
print("Attention weights shape:",A.shape)
print("Interest vectors example:\n",V[0])
print("Attention weights example:\n",A[0])

Interest vectors shape: torch.Size([64, 6, 64])
Attention weights shape: torch.Size([64, 6, 30])
Interest vectors example:
 tensor([[-1.2517e+00, -7.1477e-01,  8.9673e-01, -3.2865e-01,  5.9796e-01,
         -2.4466e-02,  1.9771e-01, -2.8867e-01, -7.3812e-01,  9.0154e-01,
          6.7274e-01,  2.3379e-01, -1.4178e-01,  2.4668e-01,  4.2556e-01,
          3.5442e-01, -1.5789e-01, -1.6583e-01, -3.1061e-01, -2.1184e-01,
          5.9121e-01, -1.1576e-01, -2.4271e-01,  4.7892e-01, -1.2787e-02,
          6.6506e-01, -1.3237e-01,  8.0141e-01,  1.5080e-01, -1.1055e+00,
          5.9046e-01, -7.6712e-01, -1.0864e-01, -3.0204e-01, -1.8062e-01,
          5.7949e-01, -1.4161e-01, -1.7392e-01,  6.9380e-01,  1.9485e-01,
          2.6011e-01, -4.3822e-01, -2.3799e-01, -1.6483e-01,  9.0329e-03,
          6.1213e-01, -5.6065e-01,  5.0541e-01,  3.1946e-01, -2.8444e-01,
         -3.6207e-01, -4.4423e-01, -3.1183e-01, -5.1761e-02, -7.4624e-02,
         -8.8902e-01, -2.9029e-01,  6.4534e-02,  3.1778e-01, -

In [132]:
loss = model.compute_loss(input_batch,target_batch) 
print("Loss:",loss.item())

Loss: 10.173213958740234


## Create Training Loop for SASRec

In [133]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") 
model = model.to(device)

In [134]:
device

device(type='mps')

In [135]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-6) 

In [159]:
for epoch in range(NUM_EPOCHS):
    model.train() 
    total_loss = 0.0 
    total_batches = 0 

    for input_batch, target_batch in train_loader:
        input_batch = input_batch.to(device) 
        target_batch = target_batch.to(device) 

        # --------- Training step -----------
        # 1. Feed Forward + 2. Compute Loss 
        loss = model.compute_loss(input_batch,target_batch)

        # 3. Reset gradients 
        optimizer.zero_grad() 

        # 4. Feed Backward
        loss.backward() 

        # 5. Update parameters
        optimizer.step() 
        # ---------- Training step ----------

        # Accumulate loss for monitoring
        total_loss += loss.item() 
        total_batches += 1
    avg_loss = total_loss / total_batches
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Average Loss: {avg_loss:.4f}")


Epoch [1/50], Average Loss: 3.9932
Epoch [2/50], Average Loss: 3.9969
Epoch [3/50], Average Loss: 3.9955
Epoch [4/50], Average Loss: 3.9879
Epoch [5/50], Average Loss: 3.9886
Epoch [6/50], Average Loss: 3.9894
Epoch [7/50], Average Loss: 3.9811
Epoch [8/50], Average Loss: 3.9896
Epoch [9/50], Average Loss: 3.9864
Epoch [10/50], Average Loss: 3.9856
Epoch [11/50], Average Loss: 3.9829
Epoch [12/50], Average Loss: 3.9767
Epoch [13/50], Average Loss: 3.9740
Epoch [14/50], Average Loss: 3.9794
Epoch [15/50], Average Loss: 3.9712
Epoch [16/50], Average Loss: 3.9698
Epoch [17/50], Average Loss: 3.9678
Epoch [18/50], Average Loss: 3.9678
Epoch [19/50], Average Loss: 3.9714
Epoch [20/50], Average Loss: 3.9612
Epoch [21/50], Average Loss: 3.9662
Epoch [22/50], Average Loss: 3.9651
Epoch [23/50], Average Loss: 3.9622
Epoch [24/50], Average Loss: 3.9590
Epoch [25/50], Average Loss: 3.9609
Epoch [26/50], Average Loss: 3.9553
Epoch [27/50], Average Loss: 3.9539
Epoch [28/50], Average Loss: 3.9565
E

## Recommend for next items  

In [160]:
TOP_K = 50 
def recommend(model, user_sequence, top_k = 10, return_attention = False): 
    model.eval() 

    if len(user_sequence) > MAX_SEQ_LEN: 
        user_sequence = user_sequence[-MAX_SEQ_LEN:] 
    padding_length = MAX_SEQ_LEN - len(user_sequence) 
    padded_sequence = [PAD_ID] * padding_length + user_sequence 

    input_tensor = torch.tensor([padded_sequence],dtype=torch.long).to(device) #[1,L] <= [L]

    with torch.no_grad():
        V, A = model(input_tensor) #[1,K,D],[1,K,L] 

        all_item_emb = model.item_embedding.weight #[num_items+1, D]
        all_item_emb = all_item_emb.T #[D, num_items+1]

        V = V.squeeze(0) #[K,D] <= [1,K,D] 

        scores_by_interest = V @ all_item_emb #[K,num_items+1] <= [K,D] @ [D,num_items+1] 

        final_scores,best_interest_for_item = torch.max(scores_by_interest,dim=0) #[num_items+1] <= max([K,num_items+1],dim=0) 
    final_scores[PAD_ID] = float("-inf")  # Mask the padding index 

    for item_id in user_sequence: 
        final_scores[item_id] = float("-inf")  # Mask already interacted items
    
    top_scores, top_items = torch.topk(final_scores, k=top_k)

    recommended_items = top_items.cpu().numpy().tolist()
    recommended_scores = top_scores.cpu().numpy().tolist()
    if return_attention:
        return recommended_items, recommended_scores, A
    else:
        return recommended_items, recommended_scores

In [174]:
example_user_id = 10
example_sequence = user_sequences[example_user_id]
recommended_items,recommended_scores = recommend(
    model=model,
    user_sequence=example_sequence,
    top_k=TOP_K
)
print("Past interacted items:\n", example_sequence)
print("Recommended items:\n ", recommended_items)
print("Recommended scores:\n ", recommended_scores)

Past interacted items:
 [82, 395, 355, 122, 41, 324, 433, 463, 99, 404, 412, 83, 312, 78, 153, 326, 37, 33, 463, 480, 148, 342, 230, 391, 27]
Recommended items:
  [333, 147, 76, 417, 478, 6, 465, 470, 362, 425, 265, 209, 358, 44, 74, 97, 25, 171, 419, 379, 293, 155, 31, 413, 167, 215, 245, 135, 141, 314, 499, 137, 297, 95, 58, 286, 399, 276, 142, 295, 77, 281, 71, 20, 384, 203, 227, 161, 356, 228]
Recommended scores:
  [9.814629554748535, 9.530057907104492, 9.491544723510742, 9.483166694641113, 9.395490646362305, 9.389684677124023, 9.212871551513672, 9.110519409179688, 9.043840408325195, 8.926614761352539, 8.887493133544922, 8.863118171691895, 8.751964569091797, 8.661685943603516, 8.615918159484863, 8.569417953491211, 8.55063247680664, 8.535064697265625, 8.492086410522461, 8.310495376586914, 8.201253890991211, 8.193558692932129, 8.180153846740723, 8.153636932373047, 8.139080047607422, 7.987306118011475, 7.981255054473877, 7.924496173858643, 7.90212345123291, 7.895782470703125, 7.830879

In [175]:
category_from_recommended = items_df[items_df['item_id'].isin(recommended_items)]['category_id'].unique()
category_from_past = items_df[items_df['item_id'].isin(example_sequence)]['category_id'].unique()
print("Categories of recommended items:\n", category_from_recommended, ",length: ", len(category_from_recommended))
print("Categories of past interacted items:\n", category_from_past, ",length: ", len(category_from_past))
recommended_categories_not_in_past = np.setdiff1d(category_from_recommended, category_from_past)
print("Recommended Categories that ARE NOT IN past interacted items:\n", recommended_categories_not_in_past, ",length: ", len(recommended_categories_not_in_past))
recommended_categories_in_past = np.intersect1d(category_from_recommended, category_from_past)
print("Recommended Categories that ARE IN past interacted items:\n", recommended_categories_in_past, ",length: ", len(recommended_categories_in_past))
length_of_total_past_categories_is_recommended = len(np.intersect1d(category_from_recommended, category_from_past))
print("Total past categories that are recommended:\n", length_of_total_past_categories_is_recommended)

Categories of recommended items:
 [9 1 8 5 7 4 6 2 0 3] ,length:  10
Categories of past interacted items:
 [9 4 8 7 5 1] ,length:  6
Recommended Categories that ARE NOT IN past interacted items:
 [0 2 3 6] ,length:  4
Recommended Categories that ARE IN past interacted items:
 [1 4 5 7 8 9] ,length:  6
Total past categories that are recommended:
 6


## Evaluation

In [163]:
def evaluate_model(model,evaluation_samples,top_k=10): 
    hits = []
    recalls = []
    ndcgs = []

    for sample in evaluation_samples:
        user_sequence = sample['input_sequence'] 
        target_item = sample['target_item'] 

        recommended_items,recommended_scores = recommend(model,user_sequence,top_k=top_k,return_attention=False)

        if target_item in recommended_items: 
            hit = 1 
            recall = 1 
            rank = recommended_items.index(target_item) + 1
            ndcg = 1 / math.log2(rank + 1)
        else:
            hit = 0 
            recall = 0 
            ndcg = 0
        hits.append(hit)
        recalls.append(recall)
        ndcgs.append(ndcg)
    hit_at_k = np.mean(hits)
    recall_at_k = np.mean(recalls)
    ndcg_at_k = np.mean(ndcgs)
    return hit_at_k, recall_at_k, ndcg_at_k,hits, recalls, ndcgs

In [166]:
hit_at_10, recall_at_10, ndcg_at_10, hits, recalls, ndcgs = evaluate_model(model,test_user_sequences,top_k=10)
print(f"Hit@10: {hit_at_10:.4f}, Recall@10: {recall_at_10:.4f}, NDCG@10: {ndcg_at_10:.4f}") 

Hit@10: 0.0900, Recall@10: 0.0900, NDCG@10: 0.0413


In [167]:
hits.count(1)

90